# 05 — RL con PPO

Entrena el modelo que usará el producto. Parte **desde cero** respecto a RL:
el único peso previo es el actor de imitación del `03` (`mlp_v1_p1s1.pt`).

- Régimen: p=1 s=1 (el ítem viene fijado; el actor elige la pose legal).
- Receta: recompensa densa de volumen por paso.
- Val de `data/scale` elige el actor. Holdout no entra.
- Exporta `artifacts/models/mlp_v1_p1s1_ppo.pt`.

Siguiente: `06_evaluar_holdout.ipynb`.

In [1]:
from pathlib import Path
import json
import sys

HERE = Path.cwd().resolve()
if HERE.name == "notebooks":
    ML_ROOT = HERE.parent
else:
    ML_ROOT = HERE if (HERE / "src_ml").is_dir() else Path("/home/edmundo/packing-services/online_policy_ml")
REPO_ROOT = ML_ROOT.parent
for p in (ML_ROOT / "src_ml", REPO_ROOT / "src"):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

from bootstrap import setup
setup()

from config import PPO_GAMMA, PPO_LR, PPO_OUTER_EPOCHS, PPO_ROLLOUTS
from evaluate import evaluate_orders, write_report
from export_ckpt import export_mlp_pt
from methodology import assert_holdout_excluded, assert_selection_split, compare_paired
from paths import BED_JSON_NAME, ORDER_IDS_NAME, REPORTS_DIR, model_path_mlp, model_path_mlp_ppo_production, scale_split_dir
from splits import load_orders
from train_ppo import fit_ppo

LOOKAHEAD_P, SELECT_S = 1, 1
init_path = model_path_mlp(LOOKAHEAD_P, SELECT_S)
out_path = model_path_mlp_ppo_production(LOOKAHEAD_P, SELECT_S)
assert init_path.is_file(), f"Falta el actor del 03: {init_path}. Corre 01→03 primero."

scale_train = scale_split_dir("train")
scale_val = scale_split_dir("val")
assert (scale_train / BED_JSON_NAME).is_file(), "Falta data/scale. Corre el 01."

ids_train = json.loads((scale_train / ORDER_IDS_NAME).read_text(encoding="utf-8"))
ids_val = json.loads((scale_val / ORDER_IDS_NAME).read_text(encoding="utf-8"))
orders = {**load_orders(scale_train / BED_JSON_NAME), **load_orders(scale_val / BED_JSON_NAME)}
assert_holdout_excluded(ids_train, role="ppo_train")
assert_holdout_excluded(ids_val, role="ppo_val")
assert_selection_split(ids_val, role="model_selection")

print("init", init_path)
print("out ", out_path)
print("train", len(ids_train), "val", len(ids_val), "epochs", PPO_OUTER_EPOCHS)

init /home/edmundo/packing-services/online_policy_ml/artifacts/models/mlp_v1_p1s1.pt
out  /home/edmundo/packing-services/online_policy_ml/artifacts/models/mlp_v1_p1s1_ppo.pt
train 80 val 20 epochs 6


In [2]:
fit = fit_ppo(
    orders,
    ids_train,
    ids_val,
    init_path=init_path,
    lookahead_p=LOOKAHEAD_P,
    select_s=SELECT_S,
    mode="place",
)
export_mlp_pt(fit["state_dict"], out_path, hidden_size=fit["hidden_size"])
print("exportado", out_path)

exportado /home/edmundo/packing-services/online_policy_ml/artifacts/models/mlp_v1_p1s1_ppo.pt


In [3]:
rows_h = evaluate_orders(orders, ids_val, engine="heuristic", lookahead_p=LOOKAHEAD_P, select_s=SELECT_S)
rows_rl = evaluate_orders(
    orders, ids_val, engine="learned", lookahead_p=LOOKAHEAD_P, select_s=SELECT_S, model_path=str(out_path)
)
cmp = compare_paired(rows_rl, rows_h, label_a="ppo", label_b="heuristic")
write_report(
    REPORTS_DIR / "05_rl_ppo.json",
    {"fit": {k: fit[k] for k in fit if k not in {"actor", "critic", "state_dict"}}, "vs_heuristic": cmp, "rows": rows_h + rows_rl},
)
print("val H", cmp["mean_b"], "PPO", cmp["mean_a"], "Δ", cmp["mean_delta"], "IC", cmp["ci_lo"], cmp["ci_hi"])
print("Siguiente: 06_evaluar_holdout.ipynb")

val H 0.67442965 PPO 0.6874582 Δ 0.01302855000000001 IC -0.0002022587499999914 0.026794220000000007
Siguiente: 06_evaluar_holdout.ipynb
